# bce-log-loss-real-fake — ex1: discriminator BCE loss = BCE(D(real), 1) + BCE(D(fake), 0)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `bce-log-loss-real-fake`. Running the final beacon cell reports progress against the `GAN: BCE log loss real/fake` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: BCE log loss real/fake` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bce-log-loss-real-fake`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bce-log-loss-real-fake"
DD_SUBTOPIC = "GAN: BCE log loss real/fake"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BCE loss for D (real + fake) — quick refresher

The discriminator is a binary classifier — its loss is the SUM of two BCE terms, one for real images (target 1), one for fakes (target 0):

```python
loss_real = F.binary_cross_entropy(D(reals), t.ones_like(D(reals)))
loss_fake = F.binary_cross_entropy(D(fakes), t.zeros_like(D(fakes)))
loss_D = loss_real + loss_fake
```

**Sum, not mean across the two terms.** Each BCE is already MEAN-reduced across the batch. Adding the two gives the discriminator twice as many gradient steps per batch effectively — the standard recipe.

**`ones_like` / `zeros_like` instead of constants.** Matches shape dtype device automatically; `t.ones(batch_size)` silently breaks if D outputs `(B, 1)` not `(B,)`.

**In practice, use `binary_cross_entropy_with_logits`.** The version that fuses sigmoid + BCE is numerically stable. The bare `F.binary_cross_entropy` (this recap) expects probabilities in `[0, 1]` — feed it sigmoid outputs, not logits.

### Exercise 1 — discriminator BCE loss = BCE(D(real), 1) + BCE(D(fake), 0)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `F.binary_cross_entropy` with `t.ones_like` / `t.zeros_like` targets to compute the discriminator's combined real-plus-fake BCE loss.
> Keywords: gan, bce, discriminator-loss, ones-zeros-like
> ```

**KCs targeted:** `bce-real-target-1`, `bce-fake-target-0-sum`

Implement `ex1_discriminator_loss(d_pred_real, d_pred_fake)`. The standard DCGAN discriminator loss:

1. `d_pred_real` are D's probability outputs on REAL images, shape `(B,)`, values in `[0, 1]` (post-sigmoid).
2. `d_pred_fake` are D's probability outputs on FAKE images, shape `(B,)`, values in `[0, 1]`.
3. Build the targets:
   - `real_targets = t.ones_like(d_pred_real)` (D wants P=1 on real)
   - `fake_targets = t.zeros_like(d_pred_fake)` (D wants P=0 on fake)
4. Compute the two BCE terms with `F.binary_cross_entropy(pred, target)`.
5. Return the SUM (not the mean) of the two terms — single scalar.

Input: `d_pred_real`, `d_pred_fake` — `(B,)` float tensors with values in `(0, 1)`.
Output: scalar tensor.

The visualization sweeps D's confidence on real images and plots the loss surface as confidence changes — convex bowl pointing at perfect classifier.

In [ ]:
def ex1_discriminator_loss(d_pred_real: Tensor, d_pred_fake: Tensor) -> Tensor:
    """BCE(D(real), 1) + BCE(D(fake), 0)."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn.functional as F
    import math

    # Perfect classifier — D=1 on real, D=0 on fake → loss ≈ 0.
    real = t.full((8,), 0.9999)
    fake = t.full((8,), 0.0001)
    loss = ex1_discriminator_loss(real, fake)
    assert loss.dim() == 0, f'loss must be scalar, got shape {tuple(loss.shape)}'
    assert loss.item() < 0.001, f'perfect classifier should give loss ~0, got {loss.item():.4f}'

    # Worst classifier — D=0 on real, D=1 on fake → loss → large.
    worst_real = t.full((8,), 0.0001)
    worst_fake = t.full((8,), 0.9999)
    loss_worst = ex1_discriminator_loss(worst_real, worst_fake)
    assert loss_worst.item() > 10.0, f'worst classifier should give large loss, got {loss_worst.item():.4f}'

    # Coin-flip classifier — D=0.5 on both → loss ≈ 2 * log(2) ≈ 1.3863.
    mid_real = t.full((8,), 0.5)
    mid_fake = t.full((8,), 0.5)
    loss_mid = ex1_discriminator_loss(mid_real, mid_fake)
    expected_mid = 2 * math.log(2)
    assert abs(loss_mid.item() - expected_mid) < 1e-4, f'coin-flip loss expected {expected_mid:.4f}, got {loss_mid.item():.4f}'

    # Numerical match against the explicit reference for a random batch.
    t.manual_seed(0)
    r = t.rand(16) * 0.8 + 0.1     # in (0.1, 0.9) to avoid log(0)
    f = t.rand(16) * 0.8 + 0.1
    got = ex1_discriminator_loss(r, f)
    expected = F.binary_cross_entropy(r, t.ones_like(r)) + F.binary_cross_entropy(f, t.zeros_like(f))
    assert t.allclose(got, expected, atol=1e-6), f'numerical mismatch: {got.item()} vs {expected.item()}'

    # Gradient flows back to D's predictions.
    r_g = t.full((4,), 0.5, requires_grad=True)
    f_g = t.full((4,), 0.5, requires_grad=True)
    ex1_discriminator_loss(r_g, f_g).backward()
    # d/dr BCE(r, 1) = -1/r so grad w.r.t. r is negative (push r toward 1).
    assert (r_g.grad < 0).all(), 'gradient on real preds should be negative (push toward 1)'
    # d/df BCE(f, 0) = 1/(1-f) so grad w.r.t. f is positive (push f toward 0).
    assert (f_g.grad > 0).all(), 'gradient on fake preds should be positive (push toward 0)'

    # --- Visualization: loss surface as D's confidence sweeps ---
    ps = t.linspace(0.01, 0.99, 99)
    # Hold fake at 0.5; sweep real.
    losses_real_sweep = [ex1_discriminator_loss(t.full((4,), p.item()), t.full((4,), 0.5)).item() for p in ps]
    # Hold real at 0.5; sweep fake.
    losses_fake_sweep = [ex1_discriminator_loss(t.full((4,), 0.5), t.full((4,), p.item())).item() for p in ps]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ps.numpy(), losses_real_sweep, label='sweep D(real); D(fake)=0.5', color='steelblue', lw=2)
    ax.plot(ps.numpy(), losses_fake_sweep, label='sweep D(fake); D(real)=0.5', color='coral', lw=2)
    ax.axvline(1.0, color='steelblue', ls=':', alpha=0.5)
    ax.axvline(0.0, color='coral', ls=':', alpha=0.5)
    ax.set_xlabel('D probability'); ax.set_ylabel('loss')
    ax.set_title('D loss vs confidence — bowl minima at correct calls')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_discriminator_loss(d_pred_real: Tensor, d_pred_fake: Tensor) -> Tensor:
    import torch.nn.functional as F
    loss_real = F.binary_cross_entropy(d_pred_real, t.ones_like(d_pred_real))
    loss_fake = F.binary_cross_entropy(d_pred_fake, t.zeros_like(d_pred_fake))
    return loss_real + loss_fake
```

**Why sum, not mean across the two terms.** Each BCE is already MEAN-reduced over the batch (`F.binary_cross_entropy` defaults to `reduction='mean'`). Summing gives D the full real-loss-plus-fake-loss signal — the convention dating back to Goodfellow 2014. Averaging would halve the discriminator's effective learning rate.

**`ones_like` / `zeros_like` are dtype/device-safe.** They inherit dtype + device from the prediction tensor — works on GPU, `bfloat16`, and weird batch shapes without breaking. The naive `t.ones(d_pred_real.shape[0])` silently breaks on GPU.

**Numerical caveat.** Bare `F.binary_cross_entropy` can produce inf when the prediction hits exactly 0 or 1. In production you'd feed logits to `F.binary_cross_entropy_with_logits` — the fused version is rock-solid numerically. This recap drills the probability-input form because that's what flashcards reference.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()